[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_03_ML_I_Recap.ipynb)

# NB Warm-up 03 — Models, estimators, and predictions

**STAT3009 · Recommender Systems**  
**Suggested time:** 10–12 minutes

> **Recall → Run → Explain.** Review the core ideas from Machine Learning I before today's lesson on validation and model selection.

**Classroom route:** quick recall (2 min) → `LinearRegression` (4 min) → familiar RS estimator (4 min) → bridge question (2 min). All data are included below; no download is needed.


## 1. Ninety-second recall

Decide whether each statement is true or false **before** opening the answers.

1. In `LinearRegression(fit_intercept=True)`, `fit_intercept` is chosen before `fit`.
2. `coef_` and `intercept_` are learned from the training examples.
3. `predict(X_new)` needs both `X_new` and its unknown answers.
4. A small **training** RMSE proves that the model will predict new rows well.

<details>
<summary><b>Reveal the answers</b></summary>

1. **True.** `fit_intercept` is a constructor setting.
2. **True.** These fitted attributes appear after `fit(X, y)`.
3. **False.** `predict` receives `X_new` only.
4. **False.** Training RMSE measures fit on rows the model learned from. We need held-out rows to check generalization.

</details>


## 2. The four questions in one familiar model

For linear regression, complete this table aloud. The answers are just below it.

| Question | Your recall |
|---|---|
| Model | What rule turns features into a prediction? |
| Hyperparameter | Which setting did we choose before fitting? |
| Fitting criterion | What error does `fit` minimize on training rows? |
| Learned parameters | Which values appear after fitting? |

<details>
<summary><b>Reveal the answers</b></summary>

- **Model:** $\widehat y=b+x^{\mathsf T}w$.
- **Hyperparameter in this example:** `fit_intercept=True`.
- **Fitting criterion:** sum of squared training errors.
- **Learned parameters:** $\widehat b$ and $\widehat w$, stored in `intercept_` and `coef_`.

</details>


## 3. Run the estimator workflow

The data follow the same contract as ML I: each row of `X_train` has one matching target in `y_train`. `X_new` contains feature rows whose answers we want to predict.


In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.utils.validation import check_is_fitted

X_train = np.array([[0.0], [1.0], [2.0], [3.0]])
y_train = np.array([1.0, 3.0, 5.0, 7.0])
X_new = np.array([[4.0], [5.0]])

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_new shape:', X_new.shape)


### Your turn — three familiar calls

Create a `LinearRegression` object with an intercept, fit it, and predict the two new rows. Use the three-line pattern from ML I; then compare with the solution.


In [ ]:
# linear_model = LinearRegression(fit_intercept=True)
# linear_model.fit(..., ...)
# new_predictions = linear_model.predict(...)
# print(new_predictions)


### Solution


In [ ]:
linear_model = LinearRegression(fit_intercept=True)
linear_model.fit(X_train, y_train)
new_predictions = linear_model.predict(X_new)

print('predictions for x=4 and x=5:', new_predictions)
print('fit_intercept:', linear_model.get_params()['fit_intercept'])
print('coef_:', linear_model.coef_)
print('intercept_:', linear_model.intercept_)


`fit_intercept` was available before training. `coef_` and `intercept_` appeared after training. The learned rule is approximately $\widehat y=1+2x$, so the new predictions should be 9 and 11.


## 4. The same interface for a recommender baseline

This is the `GlobalMeanRS` pattern from ML I. Read the two methods; you do **not** need to rewrite the class. Its model predicts one constant rating, and `fit` learns that constant from `y`.


In [ ]:
class GlobalMeanRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        self.global_mean_ = np.asarray(y).mean()
        return self

    def predict(self, X):
        check_is_fitted(self, 'global_mean_')
        return np.full(len(X), self.global_mean_)


X_ratings = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_ratings = np.array([5.0, 4.0, 3.0, 2.0])
X_pairs_new = np.array([[0, 2], [2, 0]])


### Your turn — reuse the workflow

Fit `GlobalMeanRS` on `X_ratings, y_ratings`, then predict both rows of `X_pairs_new`. Before running the code, what value should both predictions have?


In [ ]:
# rs_model = GlobalMeanRS()
# rs_model.fit(..., ...)
# rs_predictions = rs_model.predict(...)
# print(rs_predictions)


### Solution


In [ ]:
rs_model = GlobalMeanRS()
rs_model.fit(X_ratings, y_ratings)
rs_predictions = rs_model.predict(X_pairs_new)

print('learned global_mean_:', rs_model.global_mean_)
print('new-pair predictions:', rs_predictions)
print('training RMSE:', root_mean_squared_error(
    y_ratings, rs_model.predict(X_ratings)
))


## Exit ticket — bridge to Machine Learning II

`LinearRegression` and `GlobalMeanRS` have different prediction rules but share `fit(X, y)` and `predict(X_new)`. We could also fit the user-mean and item-mean estimators from ML I.

**Question:** If two estimators have different *training* RMSEs, is the smaller one necessarily better for new user–movie pairs? What kind of known ratings could help us choose **without using the final test answers**?

<details>
<summary><b>After discussion</b></summary>

No. Compare predictions on ratings that were **held out from fitting**. Today's lesson develops holdout validation and cross-validation for that choice.

</details>
